In [1]:
import pandas as pd
import torch
import torch.nn as nn
import transformers
from transformers import AutoTokenizer

In [2]:
import kagglehub

path = kagglehub.dataset_download("abhi8923shriv/sentiment-analysis-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sentiment-analysis-dataset' dataset.
Path to dataset files: /kaggle/input/sentiment-analysis-dataset


In [3]:
import os
train_file = os.path.join(path,'train.csv')
test_file = os.path.join(path,'test.csv')
train_df = pd.read_csv(train_file,encoding='latin-1')
test_df = pd.read_csv(test_file,encoding = 'latin-1')
train_df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [4]:
train_df.dropna(subset=['text', 'sentiment'], inplace=True)

In [5]:
train_df_text = train_df['selected_text'].tolist()
train_df_labels = train_df['sentiment'].tolist()
train_data = train_df_text[:int(0.8*len(train_df_text))]
train_labels = train_df_labels[:int(0.8*len(train_df_labels))]
test_data = train_df_text[int(0.8*len(train_df_text)):]
test_labels = train_df_labels[int(0.8*len(train_df_labels)):]

In [6]:
train_data[0]

'I`d have responded, if I were going'

In [7]:
mapping = {"positive": 1, "negative": 2, "neutral": 0}
train_labels = list(map(lambda x: mapping[x], train_labels))
test_labels = list(map(lambda x: mapping[x], test_labels))

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
encoded_train=tokenizer(train_data, padding=True, truncation=True, return_tensors="np")
encoded_test=tokenizer(test_data, padding=True, truncation=True, return_tensors="np")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
train_data= encoded_train['input_ids']
test_data= encoded_test['input_ids']

In [10]:
from torch.utils.data import Dataset, DataLoader
class dataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.encodings[index], self.labels[index]


In [11]:
train_dataset = DataLoader(dataset(train_data, train_labels), batch_size=32, shuffle=True)
test_dataset = DataLoader(dataset(test_data, test_labels), batch_size=32, shuffle=False)

In [12]:
class RNN(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size, output_size, pad_idx):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, input_size, padding_idx=pad_idx)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        final_memory, _ = torch.max(out, dim=1)

        return self.fc(final_memory)

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

pad_id = tokenizer.pad_token_id

my_model = RNN(vocab_size=len(tokenizer),
               input_size=128,
               hidden_size=128,
               output_size=3,
               pad_idx=pad_id).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(my_model.parameters(), lr=0.0001)
num_epochs = 5

In [14]:
my_model.train()
for epoch in range(num_epochs):
    for batch in train_dataset:
        inputs, labels = batch
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = my_model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}')

Epoch 1/5, Loss: 0.7650
Epoch 2/5, Loss: 0.5926
Epoch 3/5, Loss: 0.5025
Epoch 4/5, Loss: 0.4437
Epoch 5/5, Loss: 0.7580


In [17]:
my_model.eval()
total = 0
correct = 0
for data in test_dataset:
    inputs, labels = data
    inputs = inputs.to(device)
    labels = labels.to(device)
    outputs = my_model(inputs)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
print(f'Accuracy of the model on the test data: {100 * correct / total:.2f}%')

Accuracy of the model on the test data: 78.84%
